In [63]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


In [1]:

from pathlib import Path
import shutil
import subprocess
import builtins

ROOT = Path.cwd()

def here():
    return ROOT

def make_dir(path):
    p = ROOT / path
    p.mkdir(parents=True, exist_ok=True)
    return p

def make_file(path, content=""):
    p = ROOT / path
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding="utf-8")
    return p

def remove_path(path):
    p = ROOT / path
    if p.is_dir():
        shutil.rmtree(p)
    elif p.exists():
        p.unlink()
    return p

def run_cmd(*cmd):
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result.returncode

if __name__ == "__main__":
    print("This is a utility module. Please import and use the functions as needed.")
    make_dir("Helpers")
    make_file("Helpers/__init__.py",'''
import builtins
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Store the original input function once, globally, to avoid re-capturing a mocked input
if not hasattr(builtins, '_original_input_backup'):
    builtins._original_input_backup = builtins.input

def set_test_inputs(inputs_list):
    """
    Sets up a mock input function that draws from a list of inputs.
    If inputs_list is None or empty, restores the original input function.
    """
    if inputs_list:
        input_iter = iter(inputs_list)
        def mock_input(prompt=""):
            try:
                value = next(input_iter)
                print(f"{prompt}{value}")
                return value
            except StopIteration:
                raise EOFError("No more inputs for testing")
        builtins.input = mock_input
    else:
        builtins.input = builtins._original_input_backup

# Define test_inputs, this is where you can change your desired inputs
test_inputs = [1]  # <--- Change this list to provide your test inputs

# Apply the test inputs automatically when this cell is run
set_test_inputs(test_inputs)

def setin(*inputs):
    """
    A helper function to set test inputs for the input() function.
    Usage:
    setin("input1", "input2", "input3")
    This will set up the input() function to return "input1" on the first call,
    "input2" on the second call, and so on.
    To reset to normal input behavior, call setin() with no arguments or None:
    setin()
    """
    if inputs:
        input_iter = iter(inputs)
        def mock_input(prompt=""):
            try:
                value = next(input_iter)
                print(f"{prompt}{value}")
                return value
            except StopIteration:
                raise EOFError("No more inputs for testing")
        builtins.input = mock_input
    else:
        builtins.input = builtins._original_input_backup
              ''')
import importlib
import sys

# Remove cached module if it exists
if 'Helpers' in sys.modules:
    del sys.modules['Helpers']

# Now import fresh
import Helpers
from Helpers import setin

setin("Hello", "World")

import os
from IPython.display import display, Javascript

from pathlib import Path
import shutil
import subprocess


class WorkspaceBase:
    @staticmethod
    def get_notebook_name():
        notebook_files = [f for f in os.listdir('.') if f.endswith('.ipynb')]
        if notebook_files:
            thienotebooktitleworkspace = os.path.splitext(notebook_files[0])[0]
        else:
            thienotebooktitleworkspace = "Untitled Notebook" # Default if no .ipynb files are found
        return thienotebooktitleworkspace
    


    def get_root(self):
        return self.root

    def make_folder(self, name):
        folder = self.root / name
        folder.mkdir(parents=True, exist_ok=True)
        return folder

    def write_text(self, path, text):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding="utf-8")
        return path

    def read_text(self, path):
        return Path(path).read_text(encoding="utf-8")


    def run_cmd(self, *args):
        result = subprocess.run(args, capture_output=True, text=True)
        print(result.stdout)
        if result.stderr:
            print("Error:", result.stderr)
    def __init__(self, root=None):
        if root is None:
            root = f"{self.get_notebook_name()}_workspace"
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)

class Problem_(WorkspaceBase):
    def __init__(self, name, files=None, root=None):
        super().__init__(root=root)
        self.name = name
        self.files = files or {}
        self.folder = self.make_folder(self.safe_name())

    def safe_name(self):
        return self.name.strip().lower().replace(" ", "_")

    def add_file(self, relative_path, content):
        self.files[str(relative_path)] = content

    def get_file(self, relative_path):
        return self.files.get(str(relative_path))

    def remove_file(self, relative_path):
        self.files.pop(str(relative_path), None)

    def save(self, clear_first=False):
        if clear_first:
            self.reset_folder(self.folder)

        for relative_path, content in self.files.items():
            full_path = self.folder / relative_path
            self.write_text(full_path, content)
    def run(self, command, cwd=None):
        if cwd is None:
            cwd = self.folder
        result = subprocess.run(command, cwd=cwd, capture_output=True, text=True, shell=True)
        print(result.stdout)
        if result.stderr:
            print("Error:", result.stderr)
    def reset_folder(self, folder):
        folder = Path(folder)
        if folder.exists():
            shutil.rmtree(folder)
        folder.mkdir(parents=True, exist_ok=True)
        return folder
    
    def load(self):
        loaded = {}
        for relative_path in self.list_all_files(self.folder):
            full_path = self.folder / relative_path
            loaded[relative_path] = self.read_text(full_path)
        self.files = loaded
        return loaded

    def summary(self):
        return {
            "name": self.name,
            "folder": str(self.folder),
            "file_count": len(self.files),
            "files": sorted(self.files.keys())
        }

    def __repr__(self):
        return f"Problem(name={self.name!r}, files={len(self.files)}, folder={str(self.folder)!r})"
    
class Problem(Problem_):
    def __init__(self, name, files=None, root=None):
        super().__init__(name, files, root)
        self.run_cmd("ls", "-la", str(self.folder))
        self.reset_folder(self.folder)

    def runpy(self, command, cwd=None):
        self.run(f"python {command}", cwd)
    
    def create(self, relative_path, content):
        self.add_file(relative_path, content)
        self.save()
    def create2(self, relative_path, content):
        self.create(relative_path, content)
        self.runpy(relative_path, cwd=self.folder)



This is a utility module. Please import and use the functions as needed.


In [14]:
__ = Problem_("Example Problem")
__.add_file("main.py", """print("Hello, World!")""")
__.add_file("README.md", "# This is an example problem\n\nThis problem demonstrates the use of the Problem class to manage files and folders.")
__.save(clear_first=True)
__.summary()
__.run("python main.py")
__.reset_folder(__.folder)

Hello, World!



In [29]:
_= Problem("Example Problem 2")
_.create("main.py", """print("Hello from Problem_!")""")
_.runpy("main.py", cwd=_.folder)
_.create2("main2.py", """print("Hello from create2!")""")

total 8
drwxrwxrwx+ 2 codespace codespace 4096 Apr 28 04:52 .
drwxrwxrwx+ 4 codespace codespace 4096 Apr 28 04:52 ..

Hello from Problem_!

Hello from create2!



In [2]:
1
2


1

2